# AAI-520 Final Team Project
## Member 1 (Ahmed): Data, APIs, and Prompt Chaining

**Student:** Ahmed Eldahmy  


**GitHub:** https://github.com/Ahmedeldahmy/AAI-520-Multi-Agent-Financial-Analysis



## Progress

- Step 1: GitHub setup - Done
- Step 2: Module 4 status form - Done
- Step 3: Notebook setup - Done
- Step 4: Yahoo Finance connection - Done
- Step 5: Financial news connection - Done
- Step 6: Prompt Chaining - Done
- Step 7: Test Member 1 tools - Done
- Step 8: Run the full pipeline - Done
- Step 9: Save results - Done


# Step 3: Setup

Install the packages and load the libraries.


In [1]:
!pip -q install -U yfinance curl_cffi langgraph langchain-openai pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 10.5 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import time
import logging
import datetime
import warnings

warnings.filterwarnings("ignore")

# Hide noisy yfinance messages. Our code returns its own clean errors.
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

import pandas as pd
import yfinance as yf

from typing import Literal, Optional, TypedDict
from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph

print("Setup complete.")

Setup complete.


In [3]:
print("pandas version:", pd.__version__)
print("yfinance version:", yf.__version__)


pandas version: 2.2.3
yfinance version: 1.7.0


In [4]:
# Project settings

MAX_NEWS_ARTICLES = 5
STOCK_HISTORY_PERIOD = "1mo"

MAX_RETRIES = 3
RETRY_WAIT_SECONDS = 2

LLM_MODEL = "gpt-4o-mini"

TEST_SYMBOLS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "SPYFAKE"
]

print("Settings loaded.")

Settings loaded.


# Step 4: Yahoo Finance

This step pulls price history, company information, and recent earnings.

The price request retries automatically. Company information is handled separately so one Yahoo response does not stop the full result.

In [5]:
def _sleep_before_retry(attempt: int) -> None:
    """Wait before another Yahoo request."""
    if attempt < MAX_RETRIES:
        time.sleep(RETRY_WAIT_SECONDS)


def _download_price_history(symbol: str) -> pd.DataFrame:
    """Download recent price history with retry and fallback."""
    last_error = None

    # First try yf.download.
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            history = yf.download(
                symbol,
                period=STOCK_HISTORY_PERIOD,
                interval="1d",
                progress=False,
                auto_adjust=False,
                threads=False
            )

            if history is not None and not history.empty:
                if isinstance(history.columns, pd.MultiIndex):
                    history.columns = history.columns.get_level_values(0)

                return history

        except Exception as exc:
            last_error = exc

        _sleep_before_retry(attempt)

    # Fallback to Ticker.history.
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            ticker = yf.Ticker(symbol)

            history = ticker.history(
                period=STOCK_HISTORY_PERIOD,
                interval="1d",
                auto_adjust=False
            )

            if history is not None and not history.empty:
                return history

        except Exception as exc:
            last_error = exc

        _sleep_before_retry(attempt)

    if last_error:
        raise RuntimeError(
            f"Yahoo price request failed after retries: {last_error}"
        )

    return pd.DataFrame()

In [6]:
def fetch_stock_data(symbol: str) -> dict:
    """Get stock and company data from Yahoo Finance."""
    symbol = symbol.strip().upper()

    if not symbol:
        return {
            "symbol": "",
            "error": "Ticker is empty."
        }

    # Price data
    try:
        history = _download_price_history(symbol)
    except Exception as exc:
        return {
            "symbol": symbol,
            "error": str(exc)
        }

    if history.empty or "Close" not in history.columns:
        return {
            "symbol": symbol,
            "error": (
                f"No price data found for {symbol} "
                f"after {MAX_RETRIES} retries."
            )
        }

    close_series = history["Close"].dropna()

    if close_series.empty:
        return {
            "symbol": symbol,
            "error": f"No closing prices found for {symbol}."
        }

    current_price = round(
        float(close_series.iloc[-1]),
        2
    )

    first_price = float(close_series.iloc[0])

    price_change = None

    if first_price:
        price_change = round(
            (
                (current_price - first_price)
                / first_price
            ) * 100,
            2
        )

    # Company information is separate from price data.
    info = {}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            ticker = yf.Ticker(symbol)
            candidate = ticker.info

            if isinstance(candidate, dict):
                info = candidate
                break

        except Exception:
            pass

        _sleep_before_retry(attempt)

    # Recent earnings are optional.
    recent_earnings = {}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            ticker = yf.Ticker(symbol)
            earnings = ticker.earnings_dates

            if earnings is not None and not earnings.empty:
                row = earnings.iloc[0]

                recent_earnings = {
                    "date": str(
                        earnings.index[0].date()
                    ),
                    "eps_estimate": row.get(
                        "EPS Estimate"
                    ),
                    "reported_eps": row.get(
                        "Reported EPS"
                    ),
                    "surprise_pct": row.get(
                        "Surprise(%)"
                    ),
                }

                break

        except Exception:
            pass

        _sleep_before_retry(attempt)

    return {
        "symbol": symbol,
        "company_name": (
            info.get("longName")
            or info.get("shortName")
            or symbol
        ),
        "sector": info.get("sector", "N/A"),
        "industry": info.get("industry", "N/A"),
        "current_price": current_price,
        "price_change_30d_pct": price_change,
        "market_cap": info.get("marketCap"),
        "pe_ratio": info.get("trailingPE"),
        "revenue": info.get("totalRevenue"),
        "recent_earnings": recent_earnings,
        "error": None,
    }

In [7]:
# Test Yahoo Finance

for symbol in TEST_SYMBOLS:
    print("\n" + "=" * 50)
    print("Symbol:", symbol)

    result = fetch_stock_data(symbol)

    print(
        json.dumps(
            result,
            indent=2,
            default=str
        )
    )


Symbol: AAPL
{
  "symbol": "AAPL",
  "company_name": "Apple Inc.",
  "sector": "Technology",
  "industry": "Consumer Electronics",
  "current_price": 341.07,
  "price_change_30d_pct": 8.81,
  "market_cap": 4977636933632,
  "pe_ratio": 39.068733,
  "revenue": 466822987776,
  "recent_earnings": {
    "date": "2026-10-29",
    "eps_estimate": 1.98,
    "reported_eps": NaN,
    "surprise_pct": NaN
  },
  "error": null
}

Symbol: MSFT
{
  "symbol": "MSFT",
  "company_name": "Microsoft Corporation",
  "sector": "Technology",
  "industry": "Software - Infrastructure",
  "current_price": 516.17,
  "price_change_30d_pct": 3.99,
  "market_cap": 3832843862016,
  "pe_ratio": 28.739979,
  "revenue": 331839012864,
  "recent_earnings": {
    "date": "2026-10-28",
    "eps_estimate": 4.72,
    "reported_eps": NaN,
    "surprise_pct": NaN
  },
  "error": null
}

Symbol: NVDA
{
  "symbol": "NVDA",
  "company_name": "NVIDIA Corporation",
  "sector": "Technology",
  "industry": "Semiconductors",
  "curre

**Check**

AAPL, MSFT, and NVDA should return real stock data.

SPYFAKE should return a clean error.

Yahoo Finance is a live service, so a temporary request can still fail. This notebook retries before returning an error.

# Step 5: Financial News

This step pulls up to 5 recent news items from Yahoo Finance.

Five articles are enough for this project and keep the prompt short.

In [8]:
def fetch_financial_news(
    symbol: str,
    limit: int = MAX_NEWS_ARTICLES
) -> list:
    """Get recent financial news from Yahoo Finance."""
    symbol = symbol.strip().upper()

    if not symbol:
        return []

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            ticker = yf.Ticker(symbol)
            raw_news = ticker.news or []

            if not raw_news:
                _sleep_before_retry(attempt)
                continue

            articles = []

            for item in raw_news[:limit]:
                content = item.get(
                    "content",
                    item
                )

                provider = content.get(
                    "provider",
                    {}
                )

                if isinstance(provider, dict):
                    publisher = provider.get(
                        "displayName",
                        ""
                    )
                else:
                    publisher = str(provider)

                url_info = content.get(
                    "canonicalUrl",
                    {}
                )

                if isinstance(url_info, dict):
                    url = url_info.get(
                        "url",
                        ""
                    )
                else:
                    url = (
                        content.get("link", "")
                        or str(url_info)
                    )

                title = (
                    content.get("title", "")
                    or item.get("title", "")
                )

                description = (
                    content.get("summary", "")
                    or content.get(
                        "description",
                        ""
                    )
                    or item.get("summary", "")
                )

                if not title and not description:
                    continue

                articles.append({
                    "symbol": symbol,
                    "title": title,
                    "description": description,
                    "published_at": (
                        content.get("pubDate", "")
                        or item.get(
                            "providerPublishTime",
                            ""
                        )
                    ),
                    "publisher": (
                        publisher
                        or item.get(
                            "publisher",
                            ""
                        )
                    ),
                    "url": (
                        url
                        or item.get("link", "")
                    ),
                    "source": "Yahoo Finance",
                })

            if articles:
                return articles[:limit]

        except Exception as exc:
            last_error = exc

        _sleep_before_retry(attempt)

    if last_error:
        print(
            f"News request failed for {symbol}: "
            f"{last_error}"
        )

    return []

In [9]:
# Test AAPL news

aapl_news = fetch_financial_news("AAPL")

print("Articles found:", len(aapl_news))

for article in aapl_news[:5]:
    print("\nTitle:", article["title"])
    print("Publisher:", article["publisher"])

Articles found: 5

Title: What Actually Made Home Depot (HD) Wall Street’s Best-Kept Compounding Secret?
Publisher: Insider Monkey

Title: META, GOOGL, NVDA, BB, SPCX: Why Retail Traders Couldn’t Take Their Eyes Off These Stocks Last Week
Publisher: Stocktwits

Title: The Trust Paradox: Agent Payment Infrastructure Is Outpacing Consumer Readiness
Publisher: Forkast News

Title: Apple (AAPL)’s Premium iPhone Strategy Faces a Test Beyond Early Adopters
Publisher: Insider Monkey

Title: Apple (AAPL) Opens the Ternus Era with a Foldable iPhone. Can Premium Devices Lift Revenue?
Publisher: Insider Monkey


In [10]:
# Quick news check for all real test stocks

for symbol in ["AAPL", "MSFT", "NVDA"]:
    articles = fetch_financial_news(symbol)

    print(
        symbol,
        "- articles:",
        len(articles)
    )

    for article in articles[:2]:
        print("  -", article["title"])

AAPL - articles: 5
  - What Actually Made Home Depot (HD) Wall Street’s Best-Kept Compounding Secret?
  - META, GOOGL, NVDA, BB, SPCX: Why Retail Traders Couldn’t Take Their Eyes Off These Stocks Last Week
MSFT - articles: 5
  - The AI gold rush is fueling a record tech bond boom: Chart of the Day
  - Microsoft (MSFT) Reportedly Plans a Vast Data Center Expansion. Can Azure Justify the Cost?
NVDA - articles: 5
  - META, GOOGL, NVDA, BB, SPCX: Why Retail Traders Couldn’t Take Their Eyes Off These Stocks Last Week
  - How Investing in This Altcoin Could Make You a Millionaire


**Check**

AAPL, MSFT, and NVDA should return real news.

If a real ticker returns zero articles, stop and rerun the news step before continuing.

# Step 6: Prompt Chaining with LangGraph

Required flow:

**Ingest -> Preprocess -> Classify -> Extract -> Summarize**

The last three stages use the LLM.

Each stage uses information from the earlier stage.

In [11]:
def get_colab_secret(name: str) -> str:
    """Read a secret from Colab or the environment."""
    try:
        from google.colab import userdata

        value = userdata.get(name)

        if value:
            return value

    except Exception:
        pass

    return os.environ.get(name, "")

In [12]:
OPENAI_API_KEY = get_colab_secret(
    "OPENAI_API_KEY"
)

if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY was not found. "
        "Add it in Colab Secrets, turn on "
        "Notebook access, restart the session, "
        "and run this cell again."
    )

os.environ["OPENAI_API_KEY"] = (
    OPENAI_API_KEY
)

print("OpenAI key loaded.")

OpenAI key loaded.


In [13]:
class ClassificationResult(BaseModel):
    category: Literal[
        "earnings",
        "analyst_opinion",
        "market_news"
    ]

    confidence: Literal[
        "low",
        "medium",
        "high"
    ]


class ExtractionResult(BaseModel):
    sentiment: Literal[
        "positive",
        "neutral",
        "negative"
    ]

    key_events: list[str] = Field(
        default_factory=list
    )

    entities: list[str] = Field(
        default_factory=list
    )

    key_figures: list[str] = Field(
        default_factory=list
    )

In [14]:
class Member1State(
    TypedDict,
    total=False
):
    symbol: str

    stock_data: dict
    articles: list

    cleaned_articles: list
    cleaned_text: str

    category: str
    category_confidence: str

    sentiment: str
    key_events: list
    entities: list
    key_figures: list

    summary: str

    error: Optional[str]

In [15]:
llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0
)

classification_llm = (
    llm.with_structured_output(
        ClassificationResult
    )
)

extraction_llm = (
    llm.with_structured_output(
        ExtractionResult
    )
)

print("LLM ready.")

LLM ready.


In [16]:
def clean_text(text: str) -> str:
    """Remove simple text noise."""
    if not text:
        return ""

    text = re.sub(
        r"https?://\S+",
        "",
        text
    )

    text = re.sub(
        r"[^\w\s\.,!?%$\-]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [17]:
def node_ingest(
    state: Member1State
) -> dict:
    """Stage 1: get stock data and news."""
    symbol = state["symbol"]

    stock_data = fetch_stock_data(
        symbol
    )

    if stock_data.get("error"):
        return {
            "stock_data": stock_data,
            "articles": [],
            "error": stock_data["error"],
        }

    articles = fetch_financial_news(
        symbol
    )

    if not articles:
        return {
            "stock_data": stock_data,
            "articles": [],
            "error": (
                f"No news found for {symbol}."
            ),
        }

    return {
        "stock_data": stock_data,
        "articles": articles,
        "error": None,
    }

In [18]:
def node_preprocess(
    state: Member1State
) -> dict:
    """Stage 2: clean the news text."""
    if state.get("error"):
        return {}

    cleaned_articles = []

    for article in state.get(
        "articles",
        []
    ):
        cleaned_articles.append({
            **article,
            "clean_title": clean_text(
                article.get(
                    "title",
                    ""
                )
            ),
            "clean_description": clean_text(
                article.get(
                    "description",
                    ""
                )
            ),
        })

    cleaned_text = " ".join(
        (
            f"{article['clean_title']}. "
            f"{article['clean_description']}"
        )
        for article in cleaned_articles
    ).strip()

    if not cleaned_text:
        return {
            "cleaned_articles": (
                cleaned_articles
            ),
            "cleaned_text": "",
            "error": (
                "News text is empty "
                "after cleaning."
            ),
        }

    return {
        "cleaned_articles": (
            cleaned_articles
        ),
        "cleaned_text": cleaned_text,
    }

In [19]:
CLASSIFY_PROMPT = (
    ChatPromptTemplate.from_template(
        """
Classify the main topic of this financial news for {symbol}.

Choose one category:
- earnings
- analyst_opinion
- market_news

News:
{cleaned_text}

Use only the news provided.
"""
    )
)


def node_classify(
    state: Member1State
) -> dict:
    """Stage 3: classify the news."""
    if state.get("error"):
        return {}

    prompt = CLASSIFY_PROMPT.invoke({
        "symbol": state["symbol"],
        "cleaned_text": (
            state["cleaned_text"]
        ),
    })

    result = (
        classification_llm.invoke(
            prompt
        )
    )

    return {
        "category": result.category,
        "category_confidence": (
            result.confidence
        ),
    }

In [20]:
EXTRACT_PROMPT = (
    ChatPromptTemplate.from_template(
        """
The news for {symbol} was classified as {category}.

Use only the text below.

News:
{cleaned_text}

Extract:
1. sentiment
2. up to 5 key events
3. named entities
4. important financial figures

Do not add facts that are not in the news.
"""
    )
)


def node_extract(
    state: Member1State
) -> dict:
    """Stage 4: extract the main facts."""
    if state.get("error"):
        return {}

    prompt = EXTRACT_PROMPT.invoke({
        "symbol": state["symbol"],
        "category": state["category"],
        "cleaned_text": (
            state["cleaned_text"]
        ),
    })

    result = (
        extraction_llm.invoke(
            prompt
        )
    )

    return {
        "sentiment": result.sentiment,
        "key_events": result.key_events,
        "entities": result.entities,
        "key_figures": (
            result.key_figures
        ),
    }

In [21]:
SUMMARIZE_PROMPT = (
    ChatPromptTemplate.from_template(
        """
Write a short 2-sentence financial research summary for {symbol}.

Category: {category}
Sentiment: {sentiment}
Key events: {key_events}
Key figures: {key_figures}
Current price: {current_price}
30-day price change: {price_change}%

Use only the information provided.
"""
    )
)


def node_summarize(
    state: Member1State
) -> dict:
    """Stage 5: write the summary."""
    if state.get("error"):
        return {
            "summary": ""
        }

    stock_data = state.get(
        "stock_data",
        {}
    )

    chain = (
        SUMMARIZE_PROMPT
        | llm
        | StrOutputParser()
    )

    summary = chain.invoke({
        "symbol": state["symbol"],
        "category": state["category"],
        "sentiment": state["sentiment"],
        "key_events": (
            state["key_events"]
        ),
        "key_figures": (
            state["key_figures"]
        ),
        "current_price": stock_data.get(
            "current_price"
        ),
        "price_change": stock_data.get(
            "price_change_30d_pct"
        ),
    })

    return {
        "summary": summary.strip()
    }

In [22]:
# Build the LangGraph workflow

builder = StateGraph(
    Member1State
)

builder.add_node(
    "ingest",
    node_ingest
)

builder.add_node(
    "preprocess",
    node_preprocess
)

builder.add_node(
    "classify",
    node_classify
)

builder.add_node(
    "extract",
    node_extract
)

builder.add_node(
    "summarize",
    node_summarize
)

builder.set_entry_point(
    "ingest"
)

builder.add_edge(
    "ingest",
    "preprocess"
)

builder.add_edge(
    "preprocess",
    "classify"
)

builder.add_edge(
    "classify",
    "extract"
)

builder.add_edge(
    "extract",
    "summarize"
)

builder.add_edge(
    "summarize",
    END
)

member1_graph = builder.compile()

print(
    "Prompt Chaining ready: "
    "Ingest -> Preprocess -> "
    "Classify -> Extract -> Summarize"
)

Prompt Chaining ready: Ingest -> Preprocess -> Classify -> Extract -> Summarize


**Check**

You must see:

`OpenAI key loaded.`

and:

`Prompt Chaining ready: Ingest -> Preprocess -> Classify -> Extract -> Summarize`

# Step 7: Tools for Member 2 (Brian)

These tools let the next agent use the Member 1 output.

In [23]:
def build_data_contract(
    state: dict
) -> dict:
    """Create the output for Member 2."""
    stock_data = state.get(
        "stock_data",
        {}
    )

    return {
        "symbol": state.get("symbol"),

        "company_name": stock_data.get(
            "company_name",
            state.get("symbol")
        ),

        "category": state.get(
            "category"
        ),

        "sentiment": state.get(
            "sentiment"
        ),

        "current_price": stock_data.get(
            "current_price"
        ),

        "price_change_30d_pct": (
            stock_data.get(
                "price_change_30d_pct"
            )
        ),

        "pe_ratio": stock_data.get(
            "pe_ratio"
        ),

        "recent_earnings": stock_data.get(
            "recent_earnings",
            {}
        ),

        "entities": state.get(
            "entities",
            []
        ),

        "key_events": state.get(
            "key_events",
            []
        ),

        "key_figures": state.get(
            "key_figures",
            []
        ),

        "article_count": len(
            state.get(
                "cleaned_articles",
                []
            )
        ),

        "summary": state.get(
            "summary",
            ""
        ),

        "error": state.get(
            "error"
        ),

        "source": "Yahoo Finance",

        "date": str(
            datetime.date.today()
        ),
    }

In [24]:
@tool
def get_stock_data(
    symbol: str
) -> dict:
    """Get stock and company data."""
    return fetch_stock_data(symbol)


@tool
def get_financial_news(
    symbol: str
) -> list:
    """Get recent financial news."""
    return fetch_financial_news(
        symbol
    )


@tool
def run_member1_pipeline(
    symbol: str
) -> dict:
    """Run the Member 1 workflow."""
    final_state = (
        member1_graph.invoke({
            "symbol": (
                symbol.strip().upper()
            )
        })
    )

    return build_data_contract(
        final_state
    )


MEMBER1_TOOLS = [
    get_stock_data,
    get_financial_news,
    run_member1_pipeline,
]

print("Member 1 tools ready.")

for item in MEMBER1_TOOLS:
    print("-", item.name)

Member 1 tools ready.
- get_stock_data
- get_financial_news
- run_member1_pipeline


# Step 8: Full Test

Run the complete workflow for:

- AAPL
- MSFT
- NVDA
- SPYFAKE

The three real stocks should return full results.

SPYFAKE should return a clean error.

### Note

Yahoo Finance uses live data. News and classifications can change between runs.
That is expected because the latest articles may be different.

In [25]:
results = {}

for symbol in TEST_SYMBOLS:
    print("\n" + "=" * 60)
    print("Symbol:", symbol)

    try:
        result = (
            run_member1_pipeline.invoke({
                "symbol": symbol
            })
        )

        results[symbol] = result

        if result.get("error"):
            print("Status: error")
            print(
                "Message:",
                result["error"]
            )

        else:
            print("Status: ok")
            print(
                "Category:",
                result["category"]
            )
            print(
                "Sentiment:",
                result["sentiment"]
            )
            print(
                "Price:",
                result["current_price"]
            )
            print(
                "Articles:",
                result["article_count"]
            )
            print(
                "Entities:",
                result["entities"]
            )
            print(
                "Key events:",
                result["key_events"]
            )
            print(
                "Key figures:",
                result["key_figures"]
            )
            print(
                "Summary:",
                result["summary"]
            )

    except Exception as exc:
        results[symbol] = {
            "symbol": symbol,
            "error": str(exc)
        }

        print("Status: error")
        print("Message:", exc)


Symbol: AAPL
Status: ok
Category: market_news
Sentiment: neutral
Price: 341.07
Articles: 5
Entities: ['Apple Inc.', 'John Ternus', 'Steve Jobs', 'Zoom', 'Slack', 'Netflix']
Key events: ['Apple Inc. launched the foldable iPhone Duo on September 11, 2026.', "John Ternus debuted as Apple Inc.'s CEO on September 9, 2026, introducing the foldable iPhone Duo.", 'The foldable iPhone Duo has a premium starting price of $1,999.', 'Analysts questioned the appeal of the foldable iPhone Duo beyond wealthy tech enthusiasts.', 'Apple demonstrated the device running Zoom, Slack, and Netflix at the launch event.']
Key figures: ['$1,999']
Summary: Apple Inc. recently launched the foldable iPhone Duo, priced at $1,999, under the leadership of new CEO John Ternus. While the device showcases popular applications like Zoom, Slack, and Netflix, analysts remain skeptical about its broader market appeal beyond affluent tech enthusiasts.

Symbol: MSFT
Status: ok
Category: market_news
Sentiment: positive
Price

In [26]:
# Check the final output

required_fields = [
    "symbol",
    "company_name",
    "category",
    "sentiment",
    "current_price",
    "entities",
    "key_events",
    "key_figures",
    "article_count",
    "summary",
    "error",
    "source",
    "date",
]

check_rows = []

for symbol, result in results.items():

    if symbol == "SPYFAKE":
        passed = bool(
            result.get("error")
        )

    else:
        has_fields = all(
            field in result
            for field in required_fields
        )

        has_output = (
            result.get("category")
            in {
                "earnings",
                "analyst_opinion",
                "market_news"
            }
            and result.get("sentiment")
            in {
                "positive",
                "neutral",
                "negative"
            }
            and result.get(
                "article_count",
                0
            ) > 0
            and bool(
                result.get("summary")
            )
            and not result.get("error")
        )

        passed = (
            has_fields
            and has_output
        )

    check_rows.append({
        "Symbol": symbol,
        "Result": (
            "PASS"
            if passed
            else "CHECK"
        ),
        "Category": result.get(
            "category",
            "-"
        ),
        "Sentiment": result.get(
            "sentiment",
            "-"
        ),
        "Articles": result.get(
            "article_count",
            0
        ),
        "Error": result.get(
            "error",
            ""
        ),
    })

validation_table = pd.DataFrame(
    check_rows
)

validation_table

,Symbol,Result,Category,Sentiment,Articles,Error
0,AAPL,PASS,market_news,neutral,5,None
1,MSFT,PASS,market_news,positive,5,None
2,NVDA,PASS,market_news,positive,5,None
3,SPYFAKE,PASS,None,None,0,No price data found for SPYFAKE after 3 retries.


**Expected result**

| Symbol | Expected |
|---|---|
| AAPL | PASS |
| MSFT | PASS |
| NVDA | PASS |
| SPYFAKE | PASS |

SPYFAKE passes because the invalid ticker is handled without crashing.

# Step 9: Save Results

Save the final test output in the data folder.

In [27]:
os.makedirs(
    "data",
    exist_ok=True
)

output_file = (
    "data/member1_example_outputs.json"
)

with open(
    output_file,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        results,
        file,
        indent=2,
        default=str
    )

print("Saved:", output_file)

Saved: data/member1_example_outputs.json


In [28]:
# Show all final outputs

for symbol in TEST_SYMBOLS:
    print("\n" + "=" * 60)
    print(symbol)
    print(
        json.dumps(
            results.get(symbol, {}),
            indent=2,
            default=str
        )
    )



AAPL
{
  "symbol": "AAPL",
  "company_name": "Apple Inc.",
  "category": "market_news",
  "sentiment": "neutral",
  "current_price": 341.07,
  "price_change_30d_pct": 8.81,
  "pe_ratio": 39.068733,
  "recent_earnings": {
    "date": "2026-10-29",
    "eps_estimate": 1.98,
    "reported_eps": NaN,
    "surprise_pct": NaN
  },
  "entities": [
    "Apple Inc.",
    "John Ternus",
    "Steve Jobs",
    "Zoom",
    "Slack",
    "Netflix"
  ],
  "key_events": [
    "Apple Inc. launched the foldable iPhone Duo on September 11, 2026.",
    "John Ternus debuted as Apple Inc.'s CEO on September 9, 2026, introducing the foldable iPhone Duo.",
    "The foldable iPhone Duo has a premium starting price of $1,999.",
    "Analysts questioned the appeal of the foldable iPhone Duo beyond wealthy tech enthusiasts.",
    "Apple demonstrated the device running Zoom, Slack, and Netflix at the launch event."
  ],
  "key_figures": [
    "$1,999"
  ],
  "article_count": 5,
  "summary": "Apple Inc. recently la

## Member 1 (Ahmed) Output

The final output is ready for the next agent workflow.

It includes:

- stock data
- financial news
- category
- sentiment
- entities
- key events
- key figures
- summary
- error handling